# 02.03 语音识别与合成（ASR / TTS）

## 本节概述

<table style="text-align: left; margin-left: 0;">
<tr><td align="left"><b>前置要求</b></td><td align="left">已完成 02.02，配置好 AK/SK/project_id</td></tr>
<tr><td align="left"><b>本节目标</b></td><td align="left">用华为云 SIS 做 ASR 语音识别和 TTS 语音合成</td></tr>
<tr><td align="left"><b>本节内容</b></td><td align="left">ASR 一句话识别 → TTS 语音合成 → 多音色/语速调节</td></tr>
</table>

## 第一部分：语音识别（ASR）


In [ ]:
# 💡 如果 .env 尚未创建，请先运行 02.02 的第一个 cell 创建并填入凭证
import os, sys, json
sys.path.insert(0, os.path.abspath("./src"))   # 把 SDK 源码加入路径

from dotenv import load_dotenv
load_dotenv()

# 读取凭证（project_id 是华为云 SIS 必需的，从"我的凭证"页面获取）
ak = os.getenv('HUAWEI_SIS_AK', '')
sk = os.getenv('HUAWEI_SIS_SK', '')
region = os.getenv('HUAWEI_SIS_REGION', 'cn-east-3')
project_id = os.getenv('HUAWEI_SIS_PROJECT_ID', '')

# 安全检查
print(f"AK: {ak[:6]}..." if len(ak) > 6 else "AK: 未配置")
print(f"SK: {'已配置' if sk else '未配置'}")
print(f"Region: {region}")
print(f"Project ID: {project_id[:8]}..." if len(project_id) > 8 else "Project ID: 未配置（⚠️ SIS必需）")

# 查看 SDK 关键客户端代码结构
print("\n=== huaweicloud_sis/client/ 目录 ===")
import subprocess
print(subprocess.run(["ls", "./src/huaweicloud_sis/client/"], capture_output=True, text=True).stdout)


## 1. 调用一句话识别（SASR）

**关键 API 说明**（与 SDK 源码完全对应）：

<table style="text-align: left; margin-left: 0;">
<tr style="background-color:#f0f0f0">
  <th align="left">API</th><th align="left">正确用法</th></tr>
<tr><td align="left"><code>AsrCustomizationClient</code></td><td align="left"><code>(ak, sk, region, project_id, sis_config=config)</code>——第4个参数是 project_id</td></tr>
<tr><td align="left"><code>AsrCustomShortRequest</code></td><td align="left"><code>(audio_format, model_property, data)</code>——3个位置参数，data 是 base64 字符串</td></tr>
<tr><td align="left"><code>io_utils.encode_file</code></td><td align="left">读取音频并返回 base64 字符串（不是 read_audio_file）</td></tr>
</table>

In [ ]:
from huaweicloud_sis.client.asr_client import AsrCustomizationClient
from huaweicloud_sis.bean.asr_request import AsrCustomShortRequest
from huaweicloud_sis.exception.exceptions import ClientException, ServerException
from huaweicloud_sis.utils import io_utils
from huaweicloud_sis.bean.sis_config import SisConfig


def sasr_recognize(audio_path, audio_format='wav', language='chinese_16k_general', add_punc=True):
    """
    一句话语音识别

    参数:
        audio_path: 音频文件路径（需 16k16bit wav）
        audio_format: 音频格式 (wav/mp3/pcm)
        language: 语言属性 (chinese_16k_general 中文 / english_16k_general 英文)
        add_punc: 是否添加标点
    返回:
        识别结果 dict
    """
    # Step 1: 初始化客户端（第4个参数必须是 project_id）
    config = SisConfig()
    config.set_connect_timeout(10)
    config.set_read_timeout(10)
    asr_client = AsrCustomizationClient(ak, sk, region, project_id, sis_config=config)

    # Step 2: 构造请求（3个位置参数 + 可选 setter）
    data = io_utils.encode_file(audio_path)   # 返回 base64 字符串
    asr_request = AsrCustomShortRequest(audio_format, language, data)
    asr_request.set_add_punc('yes' if add_punc else 'no')
    asr_request.set_digit_norm('yes')          # 数字归一化为阿拉伯数字

    # Step 3: 发送请求
    result = asr_client.get_short_response(asr_request)
    return result


# 测试：识别预置音频
print("正在识别音频 ./images/16k16bit.wav ...")
try:
    result = sasr_recognize("./images/16k16bit.wav")
    print("\n✅ 识别结果:")
    print(json.dumps(result, indent=2, ensure_ascii=False))
except (ClientException, ServerException) as e:
    print(f"识别失败: {e}")
    print("💡 请检查：1) AK/SK/project_id 是否正确 2) 是否开通 SIS 服务 3) 音频格式是否为 16k16bit")
except Exception as e:
    print(f"错误: {e}")


### 识别结果解读

返回的 `result` 是 dict，关键字段：
- `result.text`：识别出的完整文字
- `result.score`：置信度（0-1）

## 2. 提取纯文本的便捷函数

ASR 返回的是 dict，多数场景我们只需要文字。封装一个便捷函数：

In [ ]:
def recognize_audio(audio_path, language='chinese_16k_general'):
    """一句话识别，返回纯文本字符串"""
    result = sasr_recognize(audio_path, language=language)
    # result 结构: {'result': {'text': '...'}, 'trace_id': '...'}
    if isinstance(result, dict) and 'result' in result:
        return result['result'].get('text', '')
    return str(result)

# 测试便捷函数
text = recognize_audio("./images/16k16bit.wav")
print(f"识别文字: {text}")


---

## 本节练习

**练习 1（选择）**：ASR 的作用是什么？
- A. 把文字转成音频
- B. 把音频转成文字
- C. 翻译不同语言
- D. 压缩音频文件

**练习 2（填空）**：华为云 SIS 一句话识别（SASR）支持的最大音频时长是 ______ 秒；调用时第 4 个必填参数是 ______（提示：从"我的凭证"获取）。

**练习 3（代码）**：如果要识别英文音频，`language` 参数应改成什么？请写出识别英文音频的调用代码。

> 💡 参考答案见下方 code cell。

In [ ]:
# 查看本节练习答案
!cat ./answer/02.03_asr_tts/answers_asr.txt



---

## 第二部分：语音合成（TTS）


In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath("./src"))
from dotenv import load_dotenv
load_dotenv()

from huaweicloud_sis.client.tts_client import TtsCustomizationClient
from huaweicloud_sis.bean.tts_request import TtsCustomRequest
from huaweicloud_sis.bean.sis_config import SisConfig

ak = os.getenv('HUAWEI_SIS_AK', '')
sk = os.getenv('HUAWEI_SIS_SK', '')
region = os.getenv('HUAWEI_SIS_REGION', 'cn-east-3')
project_id = os.getenv('HUAWEI_SIS_PROJECT_ID', '')


def tts_synthesize(text, output_path='./tts_output.wav',
                   voice='chinese_xiaoyu_common', audio_format='wav',
                   sample_rate='8000', volume=50, speed=0, pitch=0):
    """
    语音合成：把文字合成 wav 音频并保存

    参数:
        text: 待合成文本（≤500 字）
        output_path: 输出音频路径（SDK 会自动保存到这里）
        voice: 音色
        audio_format: 音频格式
        sample_rate: 采样率 (8000/16000)
        volume: 音量 0-100
        speed: 语速 -500~500
        pitch: 音高 -500~500
    返回:
        SDK 返回的结果 dict
    """
    config = SisConfig()
    config.set_connect_timeout(10)
    config.set_read_timeout(10)
    ttsc_client = TtsCustomizationClient(ak, sk, region, project_id, sis_config=config)

    # 构造请求（text 是必传的位置参数）
    ttsc_request = TtsCustomRequest(text)
    ttsc_request.set_property(voice)
    ttsc_request.set_audio_format(audio_format)
    ttsc_request.set_sample_rate(sample_rate)
    ttsc_request.set_volume(volume)
    ttsc_request.set_pitch(pitch)
    ttsc_request.set_speed(speed)
    # 关键：让 SDK 自动保存音频到文件
    ttsc_request.set_saved(True)
    ttsc_request.set_saved_path(output_path)

    result = ttsc_client.get_ttsc_response(ttsc_request)
    return result


# 测试：合成一段文字
print("正在合成语音...")
result = tts_synthesize('欢迎来到华为云语音合成体验。', './tts_demo.wav')
print(f"✅ 合成完成，结果: {result}")

# 播放验证
from IPython.display import Audio, display
display(Audio('./tts_demo.wav'))


## 1. 多音色对比

同一段文字用不同音色合成，感受差异：

In [ ]:
# 多音色对比
sample_text = '欢迎来到华为云语音合成体验。不同的音色，带来不同的感受。'
print(f"待合成文本: \"{sample_text}\"\n")

voices = [
    ('chinese_xiaoyan_common', '小妍（女声·温柔）'),
    ('chinese_xiaoyu_common', '小宇（男声·沉稳）'),
]

from IPython.display import Audio, display
for voice_id, voice_name in voices:
    output_file = f'./tts_{voice_id.split("_")[1]}.wav'
    try:
        tts_synthesize(sample_text, output_file, voice=voice_id)
        print(f"▶ {voice_name} ({voice_id}):")
        display(Audio(output_file))
    except Exception as e:
        print(f"✗ {voice_name} 合成失败: {e}")
print("\n💡 听听两者的音色差异：小妍偏柔和，小宇偏沉稳")


## 2. 语速和音高调节

`speed` 和 `pitch` 参数让语音更灵活：

In [ ]:
# 语速对比：慢速 / 正常 / 快速
speed_configs = [
    (-200, '慢速'),
    (0, '正常语速'),
    (200, '快速'),
]

text_speed_test = '华为云语音合成服务，支持语速和音高的灵活调节。'
print("语速对比测试:")
for speed_val, desc in speed_configs:
    output_file = f'./tts_speed_{speed_val}.wav'
    try:
        tts_synthesize(text_speed_test, output_file, speed=speed_val)
        print(f"\n  ▶ {desc} (speed={speed_val}):")
        from IPython.display import Audio, display
        display(Audio(output_file))
    except Exception as e:
        print(f"  ✗ {desc} 失败: {e}")


---

## 本节练习

**练习 1（选择）**：TTS 的作用是什么？
- A. 把音频转成文字
- B. 把文字转成音频
- C. 翻译不同语言
- D. 识别说话人身份

**练习 2（填空）**：华为云 TTS 中，`speed` 参数的范围是 ______ 到 ______，0 表示 ______。

**练习 3（代码）**：写一段代码，用"小妍"音色、慢速（speed=-200）合成文字"从前有座山，山里有座庙"，保存为 `story.wav` 并播放。

> 💡 参考答案见下方 code cell。

In [ ]:
# 查看本节练习答案
!cat ./answer/02.03_asr_tts/answers_tts.txt
